In [1]:
import pandas as pd
import os

In [166]:
# Function to calculate cost
def calculate_cost(row):
    # Base cost per km
    cost = row['Distance [km]'] * cost_factor_transportation
    
    # Add additional costs for Suez or Panama
    if pd.notna(row['Suez or Panama']):
        if 'Suez' in row['Suez or Panama']:
            cost += cost_factor_suez
        elif 'Panama' in row['Suez or Panama']:
            cost += cost_factor_panama
    
    return cost

# Function to create the new dataframe based on the "From" column for liquefaction cost
def create_LNG_liquefaction_df(df, factor):
    # Step 1: Create a new dataframe with unique "From" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['From'].unique(), columns=['From'])
    
    # Step 2: Adjust the "From" column by removing "_LNG" and place the original "From" values in the "To" column
    unique_from_df['To'] = unique_from_df['From']
    unique_from_df['From'] = unique_from_df['From'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

# Function to create the new dataframe based on the "To" column for regasification cost
def create_LNG_regasification_df(df, factor):
    # Step 1: Create a new dataframe with unique "To" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['To'].unique(), columns=['To'])
    
    # Step 2: Adjust the "To" column by removing "_LNG" and place the original "To" values in the "From" column
    unique_from_df['From'] = unique_from_df['To']
    unique_from_df['To'] = unique_from_df['To'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

#Define a function to multiply the distance by a cost factor for the pipelines
def pipeline_transport_cost(df, cost_factor):
    df["Cost"] = df["distance [km]"] * cost_factor
    return df

# Function to create df_supply_demand_global
def create_supply_demand_df(df, supply=True):
    # Create the new dataframe with required columns
    df_supply_demand_global = pd.DataFrame({
        'Commodity': ['Methane'] * len(df),  # Set 'Methane' for all rows
        'Node': df['Country'] + ('_Prod' if supply else ''),  # Append '_Prod' if supply is True
        'Supply': df['GWh [2020]']  # Use 'GWh [2020]' for supply
    })
    
    return df_supply_demand_global

def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Check if the input dataframe follows the (Commodity, Source, Destination) structure
    if {'Commodity', 'Source', 'Destination'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
        
    # If the input dataframe follows the (Commodity, Node, Supply) structure
    elif {'Commodity', 'Node', 'Supply'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Node']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
    
    else:
        raise ValueError("Unexpected dataframe format. Must contain either ['Commodity', 'Source', 'Destination'] or ['Commodity', 'Node', 'Supply']")

    # Fill all other columns with 0
    for col in df.columns:
        if col not in hydrogen_df.columns:  # Skip the required columns
            hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

In [167]:
hydrogen_investment=False

cost_factor_transportation = 2
cost_factor_liquefaction = 2000
cost_factor_regasification = 2000
cost_factor_panama = 2000
cost_factor_suez = 2000

cost_factor_pipelines = 0.5

## Import Data

In [4]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_data_prep')
distances_file_name = '\\distances.xlsx'

input_file_path_1  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_1))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))

In [168]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections')
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_2020')
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_2020')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_2020')
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_2020')
df_pipelines_europe = pd.read_excel(full_input_path_1, sheet_name='Connections_2021')
df_pipelines_global = pd.read_excel(full_input_path_1, sheet_name='Global_Connections')

#load input for inner-European distances
df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [169]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
df_consumption_global = df_consumption_global.iloc[:, :-2]
#gobal production
df_production_global = df_production_global.iloc[:-3]
#globale pipelines
df_pipelines_global = df_pipelines_global.iloc[:, :-5]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]

#adjust naming
df_distances = df_distances.rename(columns={"from [NUTS_ID]": "From"})
df_distances = df_distances.rename(columns={"to [NUTS_ID]": "To"})

### Demand and Supply input sheet

In [170]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

In [171]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [172]:
# Combine the demand and supply data frames
df_all_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [173]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost'] = df_LNG_global.apply(calculate_cost, axis=1)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost']]

In [174]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [175]:
LNG_liquefaction_df

,From,To,Cost
0,USA,USA_LNG,2000
1,TT,TT_LNG,2000
2,ME,ME_LNG,2000
3,QA,QA_LNG,2000
4,AF,AF_LNG,2000
5,AU,AU_LNG,2000
6,RU,RU_LNG,2000
7,EG,EG_LNG,2000
8,IM,IM_LNG,2000


In [176]:
#Pipeline cost
#calculate European pipeline cost
df_european_pipe_transport_cost = pipeline_transport_cost(df_distances, cost_factor_pipelines)
df_global_pipe_transport_cost = pipeline_transport_cost(df_pipelines_global, cost_factor_pipelines)

In [180]:
def integrate_pipeline_costs(df_pipelines, df_transport_cost):
    """
    Merges transport cost data into the pipeline dataframe based on matching From-To relationships, 
    considering both directions (From->To and To->From).
    
    Parameters:
        df_pipelines (pd.DataFrame): DataFrame containing pipeline capacities.
        df_transport_cost (pd.DataFrame): DataFrame containing transport distances and costs.
        
    Returns:
        pd.DataFrame: Updated pipeline DataFrame with an additional 'cost' column.
    """
    # Create reversed pairs for bidirectional matching (From -> To and To -> From)
    df_reversed = df_transport_cost.rename(columns={'From': 'To', 'To': 'From', 'Cost': 'Cost_reversed'})
    
    # Concatenate original and reversed cost data to handle both directions
    df_cost = pd.concat([df_transport_cost[['From', 'To', 'Cost']], df_reversed[['From', 'To', 'Cost_reversed']]], ignore_index=True)
    
    # Merge the concatenated cost data with df_pipelines to get the corresponding cost
    df_pipelines = df_pipelines.merge(
        df_cost, 
        on=['From', 'To'], 
        how='left'
    )
    
    # For cases where the reverse relation exists, use the reversed cost value
    df_pipelines['Cost'] = df_pipelines['Cost'].fillna(df_pipelines['Cost_reversed'])

    # Drop the extra reversed cost column (no longer needed)
    df_pipelines = df_pipelines.drop(columns=['Cost_reversed'])
    
    return df_pipelines

In [181]:
df_european_pipe_transport_cost

,From,To,distance [km],Cost
0,AL,EL,286.539671,143.269835
1,AL,IT,656.732394,328.366197
2,AT,DE,470.239044,235.119522
3,AT,HU,399.694418,199.847209
4,AT,IT,532.109822,266.054911
...,...,...,...,...
66,BY,UA,553.956848,276.978424
67,DZ,MA,964.864700,482.432350
68,DZ,TN,931.390627,465.695313
69,MD,UA,302.332537,151.166268


In [182]:
# Example usage:
df_pipelines_europe = integrate_pipeline_costs(df_pipelines_europe, df_european_pipe_transport_cost)

In [145]:
df_pipelines_europe

,From,To,GWh/d [capacity map],GWh/d [gasdashboard],GWh/d [transparency map],GWh/a,TWh/a,Comment,cost
0,EL,AL,487.100,NaN,NaN,177791.500000,177.791500,same as AL to IT,143.269835
1,DE,AT,344.248,NaN,NaN,125650.520000,125.650520,NaN,235.119522
2,IT,AT,193.500,NaN,NaN,70627.500000,70.627500,NaN,266.054911
3,SK,AT,1570.400,NaN,NaN,573196.000000,573.196000,NaN,207.800290
4,RS,BA,17.980,NaN,NaN,6562.700000,6.562700,NaN,120.282651
...,...,...,...,...,...,...,...,...,...
105,MD,UA,NaN,NaN,80.846504,29508.973953,29.508974,NaN,151.166268
106,BY,UA,NaN,NaN,35.531711,12969.074413,12.969074,NaN,276.978424
107,BE,UK,803.400,261.1,NaN,95301.500000,95.301500,NaN,302.582951
108,NO,UK,1499.100,1406.0,NaN,513190.000000,513.190000,NaN,697.250417


In [199]:
def create_edges_cap_cost_dataframe(df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, df_global_pipe_transport_cost, df_pipelines_europe):
    # Step 1: Collect unique From-To pairs from all dataframes
    unique_pairs = set()

    # Collecting unique pairs from each dataframe
    for df in [df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, df_global_pipe_transport_cost, df_pipelines_europe]:
        for _, row in df.iterrows():
            unique_pairs.add((row['From'], row['To']))

    # Step 2: Create the new structured dataframe with Source and Destination columns
    df_all_cost = pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

    # Step 3: Add required columns
    df_all_cost.insert(0, 'Commodity', 'Methane')  # Set commodity as Methane
    df_all_cost['initial_capacities'] = 9999  # Placeholder for capacities
    df_all_cost['max_capacities'] = 9999  # Placeholder for capacities

    # Step 4: Merge the relevant cost data from each dataframe

    # Merge with df_cost_per_km (Cost)
    df_all_cost = df_all_cost.merge(
        df_cost_per_km[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_cost_per_km'})

    # Merge with LNG_regasification_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_regasification_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_regasification'})

    # Merge with LNG_liquefaction_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_liquefaction_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_liquefaction'})

    # Merge with df_global_pipe_transport_cost (Cost and GWh/a)
    df_all_cost = df_all_cost.merge(
        df_global_pipe_transport_cost[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_global_pipe_transport'})

    # Merge with df_pipelines_europe (Cost and GWh/a for capacities)
    df_all_cost = df_all_cost.merge(
        df_pipelines_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_pipelines_europe', 'GWh/a': 'GWh_a_pipelines_europe'})

    # Step 5: Combine cost values (sum where both exist)
    df_all_cost['costs_edge'] = df_all_cost[['Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 'Cost_global_pipe_transport', 'Cost_pipelines_europe']].sum(axis=1, min_count=1)

    # Step 6: Update initial and max capacities where available
    df_all_cost['initial_capacities'] = df_all_cost['GWh_a_pipelines_europe'].fillna(df_all_cost['initial_capacities'])
    df_all_cost['max_capacities'] = df_all_cost['GWh_a_pipelines_europe'].fillna(df_all_cost['max_capacities'])

    # Step 7: Drop unnecessary columns (cost columns from individual dataframes)
    df_all_cost = df_all_cost.drop(columns=[
        'Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 'Cost_global_pipe_transport', 'Cost_pipelines_europe',
        'GWh_a_pipelines_europe'
    ])

    # Step 8: Add empty columns for future values
    df_all_cost['new_build_cost'] = 1000000
    df_all_cost['conversion_cost'] = 0
    df_all_cost['conversion_capacity_factor'] = 1

    return df_all_cost

In [200]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_all_cost = create_edges_cap_cost_dataframe(
    df_cost_per_km, 
    LNG_regasification_df, 
    LNG_liquefaction_df, 
    df_global_pipe_transport_cost, 
    df_pipelines_europe
)

# Display the result
df_edges_cap_cost

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999.0,9999.0,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999.0,9999.0,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999.0,9999.0,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999.0,9999.0,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999.0,9999.0,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
254,Methane,CR,CN,351720.0,351720.0,3689.000000,1000000,0,1
255,Methane,QA,ME,323387.0,323387.0,185.074617,1000000,0,1
256,Methane,ME_LNG,CN_LNG,9999.0,9999.0,19827.124214,1000000,0,1
257,Methane,RU_LNG,AS_LNG,9999.0,9999.0,40151.120667,1000000,0,1


In [77]:
### check for hydrogen and append with zeros if no repurpose investigation

In [85]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

# Display the result
df_edges_complete

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999,9999,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999,9999,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999,9999,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999,9999,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999,9999,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
489,Hydrogen,ES_LNG,ES,0,0,0.000000,0,0,0
490,Hydrogen,RU_LNG,PL_LNG,0,0,0.000000,0,0,0
491,Hydrogen,ME_LNG,CN_LNG,0,0,0.000000,0,0,0
492,Hydrogen,RU_LNG,AS_LNG,0,0,0.000000,0,0,0


In [78]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-7717.933443
1,Methane,AT,-83311.215033
2,Methane,BE,-166072.218600
3,Methane,BA,-9469.790436
4,Methane,BG,-28534.750500
...,...,...,...
231,Hydrogen,IN_Prod,0.000000
232,Hydrogen,AS_Prod,0.000000
233,Hydrogen,CR_Prod,0.000000
234,Hydrogen,EG_Prod,0.000000


In [19]:
df_cost_per_km.to_excel("inputs.xlsx")